# Classificação de recomendação em reviews da Steam com Jev

Neste notebook usaremos o **Jev**, modelo System One da TypeSafe AI, para responder à pergunta:

> **Com base apenas no texto da review, o usuário recomendaria ou não o jogo?**

O corpus será `_DadosLimpos/steam_reviews.pkl.gz`. O texto da coluna `review` será enviado ao modelo e a resposta será comparada com o rótulo real `voted_up` fornecido pela Steam:

- `voted_up = True` → `recomenda`;
- `voted_up = False` → `nao_recomenda`.

Ao contrário do notebook local com TF-IDF, aqui não há treinamento sobre o corpus. O Jev será avaliado em regime **zero-shot**, por meio de chamadas a uma API externa.

## 1. Abordagens comparadas

**TF-IDF + Regressão Logística**

`reviews rotuladas → treinamento local → classificador → previsão`

**Jev**

`review → state + proposição tipada → API Jev → decisão estruturada`

Usaremos a primitiva `Noul`, que avalia uma proposição binária e retorna um valor entre 0 e 1. Não usaremos uma escala de estrelas porque os dados coletados da Steam não possuem nota real de 1 a 5 para comparação.

## 2. Instalação e bibliotecas

Cada classificação realizada pelo Jev é uma chamada externa. Verifique previamente os custos, limites de uso e política de dados da API.

In [ ]:
%pip install typesafe-sdk pandas matplotlib scikit-learn

In [ ]:
import os
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from typesafe_sdk import Noul, TypeSafeClient

RANDOM_STATE = 42

## 3. Chave da API

A chave é solicitada com `getpass`, portanto não aparece na saída nem fica gravada no notebook. Se `TYPESAFE_API_KEY` já estiver definida no ambiente, ela será reutilizada.

> Nunca escreva chaves diretamente em células, comentários, arquivos versionados ou saídas do notebook. Se uma chave for exposta, revogue-a e gere outra.

In [ ]:
if not os.environ.get('TYPESAFE_API_KEY'):
    chave = getpass('Digite sua TYPESAFE_API_KEY: ').strip()
    if not chave:
        raise ValueError('A chave da API não pode ser vazia.')
    os.environ['TYPESAFE_API_KEY'] = chave

print('Chave disponível apenas no ambiente da sessão atual.')

## 4. Carregando as reviews da Steam

O código localiza a raiz do repositório mesmo quando o notebook é executado dentro de `5_Classificacao`. O pickle é carregado localmente; apenas os textos selecionados para a amostra serão enviados à API.

In [ ]:
def localizar_raiz(inicio=Path.cwd()):
    inicio = inicio.resolve()
    for pasta in (inicio, *inicio.parents):
        arquivo = pasta / '_DadosLimpos' / 'steam_reviews.pkl.gz'
        if arquivo.is_file():
            return pasta
    raise FileNotFoundError(
        'Não foi encontrado _DadosLimpos/steam_reviews.pkl.gz. '
        'Execute o notebook dentro do repositório PLN_SteamRecommend.'
    )

RAIZ = localizar_raiz()
CAMINHO_DADOS = RAIZ / '_DadosLimpos' / 'steam_reviews.pkl.gz'

reviews_completas = pd.read_pickle(CAMINHO_DADOS)
colunas_necessarias = [
    'appid', 'game_name', 'review', 'voted_up', 'review_type', 'review_idioma'
]
colunas_ausentes = set(colunas_necessarias) - set(reviews_completas.columns)
if colunas_ausentes:
    raise ValueError(f'Colunas ausentes no arquivo: {sorted(colunas_ausentes)}')

df = reviews_completas[colunas_necessarias].copy()
del reviews_completas

print('Arquivo:', CAMINHO_DADOS)
print('Dimensões:', df.shape)
df.head()

## 5. Inspeção e prevenção de vazamento

A coluna `review_type` é útil para auditoria, mas não será enviada ao Jev. Ela praticamente replica o rótulo e entregaria a resposta sem necessidade de analisar a linguagem da review.

O único conteúdo enviado como `state` será a coluna `review`.

In [ ]:
print('Valores ausentes:')
print(df.isna().sum())

print('\nIdiomas:')
print(df['review_idioma'].value_counts(dropna=False))

print('\nRótulos reais:')
print(df['voted_up'].value_counts(dropna=False))

tipo_indica_positivo = df['review_type'].eq('positive')
inconsistencias = tipo_indica_positivo.ne(df['voted_up'])
print('\nDivergências entre review_type e voted_up:', int(inconsistencias.sum()))

## 6. Preparação mínima

Como o texto será interpretado diretamente pelo Jev, não aplicaremos TF-IDF, stemming, lematização nem remoção de stopwords. Apenas normalizaremos espaços e trataremos duplicatas.

In [ ]:
df = df.dropna(subset=['review', 'voted_up']).copy()
df['review'] = (
    df['review']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
df = df[df['review'].str.len() > 0].copy()

rotulos_por_texto = df.groupby('review', sort=False)['voted_up'].transform('nunique')
df = df.loc[rotulos_por_texto.eq(1)].drop_duplicates(subset='review').copy()
df['recomendacao'] = df['voted_up'].map(
    {False: 'nao_recomenda', True: 'recomenda'}
)

print('Registros disponíveis após preparação:', len(df))
print(df['recomendacao'].value_counts())

## 7. Amostra balanceada para a API

Cada review processada representa uma chamada à API. Com o padrão de 10 exemplos por classe, a avaliação principal usa 20 chamadas. A célula de inspeção anterior à avaliação faz mais uma chamada.

A amostra é balanceada entre usuários que recomendaram e não recomendaram o jogo. Aumente o valor somente depois de considerar custo, privacidade e limites da API.

In [ ]:
AMOSTRA_POR_CLASSE = 10

amostra_jev = (
    df.groupby('recomendacao', group_keys=False)
    .sample(n=AMOSTRA_POR_CLASSE, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print('Tamanho da amostra:', len(amostra_jev))
print(amostra_jev['recomendacao'].value_counts())
amostra_jev[['game_name', 'review', 'recomendacao']].head(10)

## 8. Definindo a proposição para o Jev

`Noul` retornará um sinal entre 0 e 1 para a proposição de que o autor recomendaria o jogo. A instrução explicita que a decisão deve usar somente o texto e que críticas isoladas não significam necessariamente uma recomendação negativa.

In [ ]:
PERGUNTAS_JEV = {
    'recomendaria': Noul(
        instructions=(
            'Based only on the Steam review text, would the author recommend '
            'the video game? Consider the overall conclusion of the review, '
            'including negation, mixed opinions and explicit recommendation '
            'statements. Evaluate whether the author would select the Steam '
            'Recommended option.'
        )
    )
}

PERGUNTAS_JEV

## 9. Teste de uma chamada

Antes de processar a amostra completa, executaremos uma chamada para verificar a chave, a conexão e a estrutura da resposta. Esse texto será processado novamente na avaliação principal.

In [ ]:
texto_exemplo = amostra_jev.loc[10, 'review']

print('Texto:')
print(texto_exemplo)
print('\nRótulo real:', amostra_jev.loc[10, 'recomendacao'])

with TypeSafeClient() as client:
    resposta_exemplo = client.system_one(
        state=texto_exemplo,
        questions=PERGUNTAS_JEV,
    )

In [ ]:
prob_exemplo = float(resposta_exemplo.answers['recomendaria'].noul)
print('Sinal de recomendação retornado pelo Jev:', round(prob_exemplo, 4))
print(
    'Classe com limiar 0.5:',
    'recomenda' if prob_exemplo >= 0.5 else 'nao_recomenda',
)

## 10. Função de classificação

Usaremos inicialmente o limiar `0.5`. A confiança operacional abaixo mede apenas a distância do valor retornado até esse limiar:

`confianca = 2 × |probabilidade − 0.5|`

Essa medida varia de 0 a 1, mas não deve ser interpretada como probabilidade de acerto nem como calibração estatística.

In [ ]:
LIMIAR_RECOMENDACAO = 0.5

def classificar_com_jev(texto, client):
    resposta = client.system_one(
        state=texto,
        questions=PERGUNTAS_JEV,
    )
    prob_recomendacao = float(
        resposta.answers['recomendaria'].noul
    )
    recomendacao_prevista = (
        'recomenda'
        if prob_recomendacao >= LIMIAR_RECOMENDACAO
        else 'nao_recomenda'
    )
    confianca_limiar = 2 * abs(prob_recomendacao - 0.5)

    return {
        'jev_prob_recomendacao': prob_recomendacao,
        'jev_recomendacao': recomendacao_prevista,
        'jev_confianca_limiar': confianca_limiar,
    }

## 11. Aplicando o Jev à amostra

A execução é sequencial para facilitar o acompanhamento. Falhas ficam registradas em `erro_jev` e não interrompem os demais casos. O intervalo pode ser ajustado de acordo com os limites da API.

In [ ]:
INTERVALO_ENTRE_CHAMADAS = 0.2
resultados_api = []

with TypeSafeClient() as client:
    for indice, linha in amostra_jev.iterrows():
        try:
            resultado = classificar_com_jev(linha['review'], client)
            resultado['erro_jev'] = None
        except Exception as erro:
            resultado = {
                'jev_prob_recomendacao': None,
                'jev_recomendacao': None,
                'jev_confianca_limiar': None,
                'erro_jev': str(erro),
            }

        resultados_api.append(resultado)
        print(f'{indice + 1}/{len(amostra_jev)} concluído', end='\r')
        time.sleep(INTERVALO_ENTRE_CHAMADAS)

print('\nProcessamento concluído.')

## 12. Organizando os resultados

Juntaremos o rótulo real da Steam e a decisão do Jev. Somente chamadas concluídas com sucesso entram nas métricas.

In [ ]:
resultados_jev = pd.concat(
    [amostra_jev.reset_index(drop=True), pd.DataFrame(resultados_api)],
    axis=1,
)
resultados_validos = resultados_jev[resultados_jev['erro_jev'].isna()].copy()

print('Casos avaliados com sucesso:', len(resultados_validos), '/', len(resultados_jev))
if resultados_validos.empty:
    raise RuntimeError('Nenhuma chamada ao Jev foi concluída com sucesso.')

resultados_validos[[
    'game_name', 'review', 'recomendacao',
    'jev_prob_recomendacao', 'jev_recomendacao',
]].head()

In [ ]:
falhas = resultados_jev[resultados_jev['erro_jev'].notna()]
print('Falhas:', len(falhas))
falhas[['review', 'erro_jev']].head()

## 13. Avaliação da classificação

Calcularemos acurácia, acurácia balanceada, F1 por classe, matriz de confusão e ROC-AUC. Com apenas 20 exemplos, os números são demonstrativos e possuem grande incerteza; uma comparação rigorosa exige uma amostra maior e previamente fixada.

In [ ]:
ordem_classes = ['nao_recomenda', 'recomenda']
y_true = resultados_validos['recomendacao']
y_pred = resultados_validos['jev_recomendacao']
prob_recomenda = resultados_validos['jev_prob_recomendacao'].astype(float)

print(f'Acurácia: {accuracy_score(y_true, y_pred):.3f}')
print(f'Acurácia balanceada: {balanced_accuracy_score(y_true, y_pred):.3f}')
print(f'F1 macro: {f1_score(y_true, y_pred, average="macro"):.3f}')
print('\nRelatório por classe:\n')
print(
    classification_report(
        y_true, y_pred, labels=ordem_classes, digits=3, zero_division=0
    )
)

In [ ]:
if y_true.nunique() == 2:
    y_true_auc = y_true.eq('recomenda')
    auc = roc_auc_score(y_true_auc, prob_recomenda)
    print(f'ROC-AUC: {auc:.3f}')

    RocCurveDisplay.from_predictions(
        y_true_auc,
        prob_recomenda,
        name='Jev',
    )
    plt.title('Curva ROC — recomendação com Jev')
    plt.tight_layout()
    plt.show()
else:
    print('ROC-AUC não calculada: as chamadas válidas não contêm as duas classes.')

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=ordem_classes,
    display_labels=['Não recomenda', 'Recomenda'],
    cmap='Blues',
    values_format='d',
)
plt.title('Matriz de confusão — recomendação com Jev')
plt.tight_layout()
plt.show()

## 14. Confiança operacional e erros

Casos próximos de 0.5 são mais sensíveis ao limiar. Casos incorretos e distantes do limiar ajudam a encontrar ironia, reviews mistas, instruções inadequadas ou divergências entre o texto e a escolha registrada pelo usuário.

In [ ]:
resultados_validos[[
    'review', 'recomendacao', 'jev_recomendacao',
    'jev_prob_recomendacao', 'jev_confianca_limiar',
]].sort_values('jev_confianca_limiar').head(10)

In [ ]:
erros = resultados_validos[
    resultados_validos['recomendacao'] != resultados_validos['jev_recomendacao']
].sort_values('jev_confianca_limiar', ascending=False)

print('Quantidade de erros:', len(erros))
erros[[
    'game_name', 'review', 'recomendacao', 'jev_recomendacao',
    'jev_prob_recomendacao', 'jev_confianca_limiar',
]]

## 15. Testando reviews criadas manualmente

Os textos estão em inglês porque o corpus preparado para o projeto contém somente reviews identificadas nesse idioma. Cada texto abaixo gera uma chamada adicional à API.

In [ ]:
textos_teste = [
    'Amazing game, fun with friends and absolutely worth the price.',
    'The game crashes constantly and I cannot recommend it.',
    'It has many problems, but I still recommend buying it on sale.',
    'Beautiful graphics, terrible controls and boring gameplay.',
    'I played for hours and I am still not sure if it was worth it.',
]

resultados_teste = []
with TypeSafeClient() as client:
    for texto in textos_teste:
        resultado = classificar_com_jev(texto, client)
        resultado['review'] = texto
        resultados_teste.append(resultado)

pd.DataFrame(resultados_teste)[[
    'review', 'jev_prob_recomendacao',
    'jev_recomendacao', 'jev_confianca_limiar',
]]

## 16. TF-IDF + Regressão Logística versus Jev

| Aspecto | TF-IDF + Regressão Logística | Jev |
|---|---|---|
| Treinamento no corpus Steam | necessário | não utilizado |
| Execução | local | API externa |
| Nova tarefa | requer novo alvo/treinamento | requer nova proposição |
| Interpretabilidade | coeficientes dos termos | valor estruturado do `Noul` |
| Custo por inferência | computação local | chamada externa |
| Reprodutibilidade | versão local controlável | depende também da versão remota |
| Privacidade | texto permanece local | texto selecionado é enviado ao provedor |

Uma comparação justa deve usar exatamente o mesmo conjunto de teste para as duas abordagens e registrar as versões dos modelos e dependências.

## 17. Salvando os resultados

Salvar as respostas evita repetir chamadas pagas durante análises posteriores. O arquivo não contém a chave da API.

In [ ]:
SALVAR_RESULTADOS = True
CAMINHO_RESULTADOS = RAIZ / '5_Classificacao' / 'resultados_jev_steam.csv'

if SALVAR_RESULTADOS:
    resultados_jev.to_csv(CAMINHO_RESULTADOS, index=False)
    print('Resultados salvos em:', CAMINHO_RESULTADOS)
else:
    print('Resultados não salvos.')

## 18. Limitações e próximos passos

- Uma amostra de 20 reviews serve como demonstração, não como estimativa estável de desempenho.
- O valor do `Noul` não deve ser chamado automaticamente de probabilidade calibrada.
- A formulação da proposição influencia a tarefa operacional.
- O rótulo `voted_up` representa a escolha registrada na Steam, que pode não estar explícita no texto.
- Reviews enviadas à API deixam o ambiente local; avalie requisitos de privacidade e governança.
- Mudanças no modelo remoto podem alterar resultados futuros.

Próximos passos sugeridos:

1. aumentar e fixar a amostra de avaliação;
2. usar a mesma amostra no modelo TF-IDF local;
3. repetir o teste com diferentes formulações da proposição;
4. estudar o limiar de decisão em um conjunto de validação separado;
5. analisar erros por jogo, extensão da review e linguagem mista;
6. registrar a versão do modelo remoto quando a API disponibilizar essa informação.

## Referências

- TypeSafe AI — Introduction: https://docs.typesafe.ai/introduction
- TypeSafe AI — Quick start: https://docs.typesafe.ai/introduction/quickstart
- TypeSafe AI — Noul: https://docs.typesafe.ai/primitives/noul
- Scikit-learn — Classification metrics: https://scikit-learn.org/stable/api/sklearn.metrics.html
- Steamworks — User Reviews: https://partner.steamgames.com/doc/store/reviews